# Inspect counter-based cleaned replay files

Browse **`data/output/clean_kda`** with pandas and PyArrow, without Spark or Java.
This notebook reads saved files only. It does not rerun the cleaner, change tables,
or load the warehouse. The [cleaning walkthrough](cleaning_walkthrough.ipynb) explains
why per-tick counters replaced combat-log participant counting.

Install dependencies with `python -m pip install -r notebooks/requirements.txt` from the
repository root, select that Python environment, and run cells in order. Set
`FOCUS_MATCH_ID` to inspect one match. The all-match section checks every discovered
completed match directly from its saved files and report, not a cached batch summary.

The new output keeps the direct last hitter separate from `credited_killer_slot`,
labels assumption-dependent links with `link_status`, and retains unusual assists
with `same_side`. Exact totals do not prove every assist-to-death link.

Preview, incomplete, and candidate outputs remain explicitly labelled. Cleaner completion
means its checks passed; the warehouse migration, loader and dbt models support these fields. Rebuild the
Postgres marts after migrating and loading historical data. See [job instructions and schema additions](../docs/running-kda-clean-job.md).


In [14]:
from pathlib import Path
import json
import re
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "warehouse" / "init" / "02_clean_tables.sql").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the dota-pipeline repository.")
CLEAN_DIR = ROOT / "data" / "output" / "clean_kda"
FOCUS_MATCH_ID = 9079737909  # Use None to select the first discovered match.
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 100)

# Read the expected output schema, including credit fields, from the current
# warehouse DDL. No Spark import is required.
SQL_TYPES = {"BIGINT": pa.int64(), "INTEGER": pa.int32(), "DOUBLE PRECISION": pa.float64(),
             "TEXT": pa.string(), "BOOLEAN": pa.bool_()}
specs = {}
sql = (ROOT / "warehouse" / "init" / "02_clean_tables.sql").read_text()
for table, body in re.findall(r"CREATE TABLE IF NOT EXISTS clean\.(\w+)\s*\((.*?)\);", sql, flags=re.S):
    columns = []; inline_keys = []
    for line in body.splitlines():
        match = re.match(r"\s*(\w+)\s+(BIGINT|INTEGER|DOUBLE PRECISION|TEXT|BOOLEAN)\b(.*)", line)
        if match:
            name, dtype, constraints = match.groups()
            columns.append({"column": name, "sql_type": dtype,
                            "required": "NOT NULL" in constraints or "PRIMARY KEY" in constraints})
            if "PRIMARY KEY" in constraints:
                inline_keys.append(name)
    explicit_key = re.search(r"PRIMARY KEY\s*\(([^)]+)\)", body)
    key = [c.strip() for c in explicit_key.group(1).split(",")] if explicit_key else inline_keys
    for column in columns:
        column["required"] = column["required"] or column["column"] in key
    specs[table] = {"columns": columns, "key": key}
if len(specs) != 9:
    raise RuntimeError("Expected nine warehouse clean-table definitions; inspect the DDL parser.")
CREDIT_METHOD = "per_tick_counters"
CONTRACT_VERSION = "kda-credit-v1"
for table, required in {"deaths": {"credited_killer_slot", "link_status"},
                        "death_assists": {"same_side"}}.items():
    actual = {column["column"] for column in specs[table]["columns"]}
    if not required.issubset(actual):
        raise RuntimeError(f"Warehouse DDL is missing counter-credit fields: {table}")
OPTIONAL_EMPTY = {"purchases", "death_assists"}

## 1. Find outputs and check completion status

Reports, completion markers, errors, preview flags, credit method and contract version
must agree before a match is treated as completed counter-based output. Matches with
only review or candidate data are also listed. File and K/D/A checks appear below.


In [15]:
match_ids = set()
for folder in [*(CLEAN_DIR / t for t in specs), CLEAN_DIR / "_review", CLEAN_DIR / "_complete"]:
    for p in folder.glob("match_id=*"):
        suffix = p.name.split("=", 1)[1]
        if suffix.isdigit():
            match_ids.add(int(suffix))
match_ids = sorted(match_ids)
if not match_ids:
    raise RuntimeError(f"No clean outputs found under {CLEAN_DIR}. Run the cleaner first.")
reports = {}; run_rows = []
for mid in match_ids:
    path = CLEAN_DIR / "_review" / f"match_id={mid}" / "report.json"
    report = json.loads(path.read_text()) if path.exists() else None
    if report is not None and report.get("match_id") != mid:
        raise ValueError(f"Report match ID disagrees with its folder: {path}")
    reports[mid] = report
    marker = CLEAN_DIR / "_complete" / f"match_id={mid}"
    marker_present = marker.exists() and any(p.is_file() for p in marker.rglob("*"))
    ready = bool(marker_present and report and report.get("complete") is True
                 and report.get("preview") is False and report.get("errors") == []
                 and report.get("credit_method") == CREDIT_METHOD
                 and report.get("contract_version") == CONTRACT_VERSION)
    run_rows.append({"match_id": mid, "report_found": report is not None,
        "preview": report.get("preview") if report else None,
        "report_complete": report.get("complete") if report else None,
        "marker_present": marker_present,
        "credit_method": report.get("credit_method") if report else None,
        "contract_version": report.get("contract_version") if report else None,
        "ready": ready,
        "errors": "; ".join(report.get("errors", [])) if report else "Report missing"})
run_status = pd.DataFrame(run_rows)
display(run_status)
MATCH_ID = FOCUS_MATCH_ID if FOCUS_MATCH_ID is not None else match_ids[0]
if MATCH_ID not in match_ids:
    raise ValueError(f"Match {MATCH_ID} has no output. Choose one from {match_ids}.")
report = reports[MATCH_ID]
print("Selected match:", MATCH_ID)
print("Report:", CLEAN_DIR / "_review" / f"match_id={MATCH_ID}" / "report.json")
SELECTED_READY = bool(run_status.loc[run_status["match_id"].eq(MATCH_ID), "ready"].iloc[0])


,match_id,report_found,preview,report_complete,marker_present,credit_method,contract_version,ready,errors
0,9079737909,True,False,True,True,per_tick_counters,kda-credit-v1,True,
1,9079737994,True,False,True,True,per_tick_counters,kda-credit-v1,True,
2,9079738443,True,False,True,True,per_tick_counters,kda-credit-v1,True,
3,9079738574,True,False,True,True,per_tick_counters,kda-credit-v1,True,
4,9079740225,True,False,True,True,per_tick_counters,kda-credit-v1,True,
5,9079740632,True,False,True,True,per_tick_counters,kda-credit-v1,True,
6,9079741152,True,False,True,True,per_tick_counters,kda-credit-v1,True,
7,9079743269,True,False,True,True,per_tick_counters,kda-credit-v1,True,
8,9079745340,True,False,True,True,per_tick_counters,kda-credit-v1,True,
9,9079746862,True,False,True,True,per_tick_counters,kda-credit-v1,True,


Selected match: 9079737909
Report: /Users/maximus/Downloads/SCHOOL/IS3107/dota-pipeline/data/output/clean_kda/_review/match_id=9079737909/report.json


## 2. Load tables and distinguish written output from review candidates

Parquet files physically omit `match_id` because it is in the partition path. The reader restores
it as a 64-bit integer. Reading physical files directly avoids accidentally mixing ancestor and
candidate partition names. Expected empty optional tables are represented as empty typed frames;
missing required tables are flagged, not silently assumed valid.

In [16]:
def read_parts(folder, match_id=None):
    files = sorted(folder.glob("*.parquet"))
    if not files:
        return None
    parts = []
    for path in files:
        frame = pq.ParquetFile(path).read().to_pandas().convert_dtypes()
        if match_id is not None:
            if "match_id" in frame and not frame["match_id"].eq(match_id).all():
                raise ValueError(f"Unexpected embedded match ID: {path}")
            frame["match_id"] = pd.Series(match_id, index=frame.index, dtype="Int64")
        parts.append(frame)
    return pd.concat(parts, ignore_index=True)

def empty_table(table):
    schema = pa.schema([(c["column"], SQL_TYPES[c["sql_type"]]) for c in specs[table]["columns"]])
    return pa.Table.from_batches([], schema=schema).to_pandas().convert_dtypes()

tables = {}; sources = {}; folders = {}; table_rows = []
for table in specs:
    regular = CLEAN_DIR / table / f"match_id={MATCH_ID}"
    candidate = CLEAN_DIR / "_review" / f"match_id={MATCH_ID}" / "candidates" / table / f"match_id={MATCH_ID}"
    expected = report.get("table_rows", {}).get(table) if report else None
    df = read_parts(regular, MATCH_ID)
    source = "published output" if SELECTED_READY else "written output - incomplete/unverified"
    folder = regular
    if df is None:
        df = read_parts(candidate, MATCH_ID)
        source = "CANDIDATE - review only"; folder = candidate
    if df is None and table in OPTIONAL_EMPTY and expected == 0:
        df = empty_table(table)
        source = "empty published table" if SELECTED_READY else "empty output - incomplete/unverified"
    if df is None:
        source = "MISSING"
    else:
        tables[table] = df; sources[table] = source; folders[table] = folder
    table_rows.append({"table": table, "source": source,
        "loaded_rows": len(df) if df is not None else None, "reported_rows": expected,
        "count_matches_report": len(df) == expected if df is not None and expected is not None else None})
table_status = pd.DataFrame(table_rows)
display(table_status)
print("Selected match:", "completed counter-based output" if SELECTED_READY else "incomplete/unverified output")
print("Missing tables, row-count differences and schema problems remain visible below.")

,table,source,loaded_rows,reported_rows,count_matches_report
0,matches,published output,1,1,True
1,players,published output,10,10,True
2,player_stats,published output,15080,15080,True
3,positions,published output,14570,14570,True
4,damage_events,published output,8783,8783,True
5,deaths,published output,37,37,True
6,death_assists,published output,26,26,True
7,purchases,published output,200,200,True
8,final_inventory,published output,75,75,True


Selected match: completed counter-based output
Missing tables, row-count differences and schema problems remain visible below.


## 3. First five rows of every available table

Candidate deaths and assists are explicitly labelled in each preview. Change the selected match
in setup and rerun to inspect another run.

In [17]:
for table, df in tables.items():
    print(f"\n{table}: {sources[table]} | {len(df):,} rows")
    order = [c for c in ["tick", "player_slot", "victim_slot", "inventory_slot"] if c in df]
    display((df.sort_values(order, kind="stable") if order else df).head(5))


matches: published output | 1 rows


,game_mode,winner_team,end_time,game_start_tick,duration_s,match_id
0,22,3,1794045663,4260,1222.066711,9079737909



players: published output | 10 rows


,player_slot,team,team_slot,hero_name,hero_id,match_id
0,0,2,0,npc_dota_hero_juggernaut,8,9079737909
1,1,2,1,npc_dota_hero_legion_commander,104,9079737909
2,2,2,2,npc_dota_hero_tidehunter,29,9079737909
3,3,2,3,npc_dota_hero_axe,2,9079737909
4,4,2,4,npc_dota_hero_sniper,35,9079737909



player_stats: published output | 15,080 rows


,player_slot,tick,game_time_s,level,kills,deaths,assists,net_worth,unspent_gold,total_earned_gold,total_earned_xp,last_hits,denies,hero_damage,tower_damage,healing,match_id
0,0,30,-141,0,0,0,0,0,600,0,0,0,0,0,0,0,9079737909
1,1,30,-141,0,0,0,0,0,600,0,0,0,0,0,0,0,9079737909
2,2,30,-141,0,0,0,0,0,600,0,0,0,0,0,0,0,9079737909
3,3,30,-141,0,0,0,0,0,600,0,0,0,0,0,0,0,9079737909
4,4,30,-141,0,0,0,0,0,600,0,0,0,0,0,0,0,9079737909


... (8 outputs left out)


purchases: published output | 200 rows


,tick,game_time_s,player_slot,item_name,quantity,match_id
21,4256,-0.133333,1,item_quelling_blade,1,9079737909
45,4256,-0.133333,3,item_quelling_blade,1,9079737909
14,4256,-0.133333,4,item_tango,1,9079737909
76,4258,-0.066667,0,item_quelling_blade,1,9079737909
77,4258,-0.066667,5,item_tango,1,9079737909



final_inventory: published output | 75 rows


,player_slot,inventory_slot,item_name,match_id
53,0,0,item_bfury,9079737909
0,0,1,item_magic_wand,9079737909
12,0,2,item_foragers_mana,9079737909
1,0,3,item_power_treads,9079737909
34,0,4,item_wraith_band,9079737909


## 4. Column types, missing required fields, and repeated keys

Checks use the current warehouse DDL, which includes `deaths.credited_killer_slot`,
`deaths.link_status`, and `death_assists.same_side`.
A null credited killer is allowed when no player received kill credit; the direct
last hitter may also have no player slot. Link status and same-side flags must be present.

For saved files, types are checked against physical Arrow schemas. Structural checks
alone do not establish credit accuracy. The independent K/D/A recount follows below.


In [18]:
checks = []; type_rows = []
for table, df in tables.items():
    spec = specs[table]
    expected_cols = [c["column"] for c in spec["columns"]]
    missing = sorted(set(expected_cols) - set(df.columns))
    key_present = all(c in df for c in spec["key"])
    required = [c["column"] for c in spec["columns"] if c["required"] and c["column"] in df]
    repeated = df.groupby(spec["key"], dropna=False).size().gt(1).sum() if key_present else None
    checks.append({"table": table, "source": sources[table], "missing_columns": ", ".join(missing),
        "rows_with_required_nulls": int(df[required].isna().any(axis=1).sum()),
        "repeated_key_groups": repeated,
        "empty_required_table": df.empty and table not in OPTIONAL_EMPTY,
        "invalid_link_statuses": int((~df["link_status"].isin(
            ["exact", "participant_assumed", "same_side_possible"])).sum())
            if table == "deaths" and "link_status" in df else None})
    for path in sorted(folders[table].glob("*.parquet")):
        schema = pq.ParquetFile(path).schema_arrow
        for c in spec["columns"]:
            name = c["column"]
            if name == "match_id":
                actual = "partition metadata"; ok = name not in schema.names
            else:
                field = schema.field(name) if name in schema.names else None
                actual = str(field.type) if field is not None else "MISSING"
                ok = field is not None and field.type.equals(SQL_TYPES[c["sql_type"]])
            type_rows.append({"table": table, "column": name, "sql_type": c["sql_type"],
                              "physical_type": actual, "matches": ok})
quality = pd.DataFrame(checks)
types = pd.DataFrame(type_rows).drop_duplicates()
display(quality)
print("Physical type mismatches:")
display(types.loc[~types["matches"]] if not types.empty else types)
TABLE_TO_DESCRIBE = "player_stats"
display(types.loc[types["table"].eq(TABLE_TO_DESCRIBE)] if not types.empty else types)

,table,source,missing_columns,rows_with_required_nulls,repeated_key_groups,empty_required_table,invalid_link_statuses
0,matches,published output,,0,0,False,NaN
1,players,published output,,0,0,False,NaN
2,player_stats,published output,,0,0,False,NaN
3,positions,published output,,0,0,False,NaN
4,damage_events,published output,,0,0,False,NaN
5,deaths,published output,,0,0,False,0.0
6,death_assists,published output,,0,0,False,NaN
7,purchases,published output,,0,0,False,NaN
8,final_inventory,published output,,0,0,False,NaN


Physical type mismatches:


,table,column,sql_type,physical_type,matches


,table,column,sql_type,physical_type,matches
12,player_stats,match_id,BIGINT,partition metadata,True
13,player_stats,player_slot,INTEGER,int32,True
14,player_stats,tick,INTEGER,int32,True
15,player_stats,game_time_s,DOUBLE PRECISION,double,True
16,player_stats,level,INTEGER,int32,True
17,player_stats,kills,INTEGER,int32,True
18,player_stats,deaths,INTEGER,int32,True
19,player_stats,assists,INTEGER,int32,True
20,player_stats,net_worth,INTEGER,int32,True
21,player_stats,unspent_gold,INTEGER,int32,True


## 5. Credit labels and saved review rows

A blocking finding with zero affected rows is a pass. Nonempty diagnostic files are
saved under `_review`. `scoreboard_disagreements` counts players, not proven incorrect
events. `kda_assumed_links`, `kda_same_side_assists`, and `kda_credit_review` retain
labels or excluded/merged-event notes even when a match completes successfully.

The default finding below selects a nonempty credit diagnostic when available. Change
`REVIEW_FINDING` to browse another. Last-hitter differences and same-side credit are
also read directly from the loaded tables, with their published/candidate source shown.


In [19]:
findings = pd.DataFrame([{"finding": name, **details}
    for name, details in (report.get("findings", {}) if report else {}).items()])
if not findings.empty:
    findings["blocks_this_run"] = findings["blocking"] & findings["rows"].gt(0)
display(findings)
review_dir = CLEAN_DIR / "_review" / f"match_id={MATCH_ID}"
nonempty_findings = [name for name, detail in (report.get("findings", {}) if report else {}).items()
                     if detail.get("rows", 0) > 0]
REVIEW_FINDING = next((name for name in ["kda_assumed_links", "kda_same_side_assists", "kda_credit_review"]
                       if name in nonempty_findings), "scoreboard_disagreements")
review_rows = read_parts(review_dir / REVIEW_FINDING)
if review_rows is None:
    print(f"No saved diagnostic rows for {REVIEW_FINDING}.")
else:
    display(review_rows)
if "deaths" in tables and {"credited_killer_slot", "link_status"}.issubset(tables["deaths"].columns):
    deaths = tables["deaths"]
    print("Assumption-labelled deaths:", sources["deaths"])
    display(deaths.loc[deaths["link_status"].ne("exact")])
    print("Last hitter differs from credited killer:", sources["deaths"])
    display(deaths.loc[deaths["credited_killer_slot"].notna()
        & (deaths["killer_slot"].isna()
           | deaths["killer_slot"].ne(deaths["credited_killer_slot"]).fillna(False)),
        ["match_id", "tick", "victim_slot", "killer_name", "killer_slot", "credited_killer_slot", "link_status"]])
if "death_assists" in tables and "same_side" in tables["death_assists"]:
    print("Same-side assists:", sources["death_assists"])
    display(tables["death_assists"].loc[tables["death_assists"]["same_side"].fillna(False)])


,finding,rows,blocking,blocks_this_run
0,invalid_game_clock,0,True,False
1,excluded_resource_rows,0,False,False
2,duplicate_player_snapshots,0,True,False
3,unmapped_resource_rows,0,True,False
4,duplicate_team_snapshots,0,True,False
5,missing_team_snapshots,0,True,False
6,unknown_replication_marker,0,True,False
7,invalid_hero_player_ids,0,True,False
8,unmapped_hero_teams,0,True,False
9,damage_unknown_illusion_flags,0,True,False


No saved diagnostic rows for scoreboard_disagreements.
Assumption-labelled deaths: published output


,tick,game_time_s,victim_slot,killer_slot,killer_name,credited_killer_slot,link_status,match_id


Last hitter differs from credited killer: published output


,match_id,tick,victim_slot,killer_name,killer_slot,credited_killer_slot,link_status
23,9079737909,17682,8,npc_dota_goodguys_tower1_mid,<NA>,4,exact
29,9079737909,19898,8,npc_dota_goodguys_tower1_mid,<NA>,4,exact


Same-side assists: published output


,tick,victim_slot,assist_slot,same_side,match_id


## 6. Recount K/D/A from the actual loaded credit tables

Always calculate the counts below, even when no disagreement diagnostic exists.
Compare each player's latest written `player_stats` snapshot with kills counted by
**`credited_killer_slot`**, deaths by `victim_slot`, and assists by `assist_slot`.
The sign of each difference is credited count minus final snapshot value.

For an incomplete run this can inspect candidate tables; the source and completion
status stay visible. It does not make those candidates published output. This check
compares clean outputs with clean snapshots; the walkthrough separately checks the raw
counter history and final raw scoreboard. Neither is an external Steam reconciliation.


In [20]:
def compare_credit(roster, player_stats, deaths, assists):
    keys = ["match_id", "player_slot"]
    if roster[keys].isna().any(axis=None) or roster.duplicated(keys).any():
        raise ValueError("The player roster has null or repeated keys")
    last_ticks = player_stats.groupby(keys, as_index=False)["tick"].max()
    final = player_stats.merge(last_ticks, on=[*keys, "tick"], how="inner", validate="many_to_one")
    if final.duplicated(keys).any():
        raise ValueError("Final player snapshots are ambiguous; inspect repeated keys")
    comparison = roster[[*keys, "hero_name"]].merge(
        final[[*keys, "kills", "deaths", "assists"]], on=keys,
        how="outer", validate="one_to_one", indicator="roster_status")
    for events, slot, stat in [(deaths, "credited_killer_slot", "kills"),
                               (deaths, "victim_slot", "deaths"),
                               (assists, "assist_slot", "assists")]:
        if events[slot].isna().any() and slot != "credited_killer_slot":
            raise ValueError(f"Missing required credit slot: {slot}")
        credit = events.dropna(subset=[slot])
        # Do not silently lose credited players absent from the roster.
        membership = credit[["match_id", slot]].rename(columns={slot: "player_slot"}).merge(
            roster[keys].assign(in_roster=True), on=keys, how="left", validate="many_to_one")
        if membership["in_roster"].isna().any():
            raise ValueError(f"Credit references a player outside the roster: {slot}")
        counts = credit.groupby(["match_id", slot]).size().rename(f"credited_{stat}").reset_index()
        counts = counts.rename(columns={slot: "player_slot"})
        comparison = comparison.merge(counts, on=keys, how="left", validate="one_to_one")
        comparison[f"credited_{stat}"] = comparison[f"credited_{stat}"].fillna(0).astype("Int64")
        comparison[f"{stat}_difference"] = comparison[f"credited_{stat}"] - comparison[stat]
    differences = [f"{stat}_difference" for stat in ["kills", "deaths", "assists"]]
    comparison["mismatch"] = (comparison[differences].isna().any(axis=1)
        | comparison[differences].ne(0).fillna(True).any(axis=1)
        | comparison["roster_status"].ne("both"))
    return comparison

comparison = None
needed = {"players", "player_stats", "deaths", "death_assists"}
if not needed.issubset(tables):
    print("Cannot recount credit; missing:", sorted(needed - set(tables)))
elif "credited_killer_slot" not in tables["deaths"]:
    print("Old death schema: run the counter-based cleaner first.")
else:
    comparison = compare_credit(tables["players"], tables["player_stats"],
                                tables["deaths"], tables["death_assists"])
    print("Credit sources:", sources["deaths"], "|", sources["death_assists"])
    print("Completed run:", SELECTED_READY)
    display(comparison.sort_values("player_slot"))
    print("Players checked:", len(comparison), "| K/D/A values:", len(comparison) * 3,
          "| Mismatching players:", int(comparison["mismatch"].sum()))
    display(comparison.loc[comparison["mismatch"]])


Credit sources: published output | published output
Completed run: True


,match_id,player_slot,hero_name,kills,deaths,assists,roster_status,credited_kills,kills_difference,credited_deaths,deaths_difference,credited_assists,assists_difference,mismatch
0,9079737909,0,npc_dota_hero_juggernaut,1,3,5,both,1,0,3,0,5,0,False
1,9079737909,1,npc_dota_hero_legion_commander,3,3,4,both,3,0,3,0,4,0,False
2,9079737909,2,npc_dota_hero_tidehunter,4,2,3,both,4,0,2,0,3,0,False
3,9079737909,3,npc_dota_hero_axe,2,3,4,both,2,0,3,0,4,0,False
4,9079737909,4,npc_dota_hero_sniper,2,8,0,both,2,0,8,0,0,0,False
5,9079737909,5,npc_dota_hero_drow_ranger,1,4,5,both,1,0,4,0,5,0,False
6,9079737909,6,npc_dota_hero_sven,2,5,2,both,2,0,5,0,2,0,False
7,9079737909,7,npc_dota_hero_centaur,2,3,2,both,2,0,3,0,2,0,False
8,9079737909,8,npc_dota_hero_huskar,6,2,0,both,6,0,2,0,0,0,False
9,9079737909,9,npc_dota_hero_lina,5,4,1,both,5,0,4,0,1,0,False


Players checked: 10 | K/D/A values: 30 | Mismatching players: 0


,match_id,player_slot,hero_name,kills,deaths,assists,roster_status,credited_kills,kills_difference,credited_deaths,deaths_difference,credited_assists,assists_difference,mismatch


## 7. Verify all discovered completed matches

This summary reads published files for every match whose report and marker indicate
counter-based completion. It checks saved row counts and physical schemas for all nine
tables, then recounts K/D/A for every player. Incomplete runs are shown as unverified;
candidate files are not substituted here. Errors are listed instead of silently passing.

Assumption-dependent links and same-side assists remain visible alongside exact totals.
This section recalculates its results each time it runs and does not read `validation.json`.


In [21]:
all_match_rows = []
for mid in match_ids:
    ready = bool(run_status.loc[run_status["match_id"].eq(mid), "ready"].iloc[0])
    summary = {"match_id": mid, "cleaner_complete": ready, "files_match_report": False,
               "schema_matches": False, "players_checked": None, "kda_mismatching_players": None,
               "assumed_death_links": None, "same_side_assists": None,
               "verified": False, "inspection_error": ""}
    if not ready:
        summary["inspection_error"] = "Not a completed counter-based run; inspect its report/candidates"
        all_match_rows.append(summary)
        continue
    try:
        current_report = reports[mid]
        counts_ok, schema_ok = True, True
        for table, spec in specs.items():
            files = sorted((CLEAN_DIR / table / f"match_id={mid}").glob("*.parquet"))
            count = sum(pq.ParquetFile(p).metadata.num_rows for p in files)
            counts_ok &= count == current_report.get("table_rows", {}).get(table)
            counts_ok &= bool(count or table in OPTIONAL_EMPTY)
            for p in files:
                schema = pq.ParquetFile(p).schema_arrow
                schema_ok &= "match_id" not in schema.names
                for column in spec["columns"]:
                    name = column["column"]
                    if name != "match_id":
                        schema_ok &= (name in schema.names
                            and schema.field(name).type.equals(SQL_TYPES[column["sql_type"]]))
        summary["files_match_report"], summary["schema_matches"] = bool(counts_ok), bool(schema_ok)
        selected = {}
        for table in ["players", "player_stats", "deaths", "death_assists"]:
            frame = read_parts(CLEAN_DIR / table / f"match_id={mid}", mid)
            if frame is None and table in OPTIONAL_EMPTY and current_report["table_rows"].get(table) == 0:
                frame = empty_table(table)
            if frame is None:
                raise ValueError(f"Missing published {table}")
            selected[table] = frame
        credited = compare_credit(selected["players"], selected["player_stats"],
                                   selected["deaths"], selected["death_assists"])
        labels = selected["deaths"]["link_status"]
        sides = selected["death_assists"]["same_side"]
        if not labels.isin(["exact", "participant_assumed", "same_side_possible"]).all() or sides.isna().any():
            raise ValueError("Missing or invalid credit labels")
        if selected["deaths"].duplicated(specs["deaths"]["key"]).any():
            raise ValueError("Repeated published death key")
        if selected["death_assists"].duplicated(specs["death_assists"]["key"]).any():
            raise ValueError("Repeated published assist key")
        summary.update(players_checked=len(credited),
                       kda_mismatching_players=int(credited["mismatch"].sum()),
                       assumed_death_links=int(labels.ne("exact").sum()),
                       same_side_assists=int(sides.sum()))
        summary["verified"] = bool(counts_ok and schema_ok and len(credited) == 10
                                   and not credited["mismatch"].any())
    except (KeyError, ValueError, TypeError, OSError) as exc:
        summary["inspection_error"] = str(exc)
    all_match_rows.append(summary)
all_match_summary = pd.DataFrame(all_match_rows)
display(all_match_summary)
print("Matches with verified published credit:", int(all_match_summary["verified"].sum()), "/", len(match_ids))
print("Players checked:", int(all_match_summary["players_checked"].fillna(0).sum()))
print("Mismatching players:", int(all_match_summary["kda_mismatching_players"].fillna(0).sum()),
      "| Not verified:", int((~all_match_summary["verified"]).sum()))


,match_id,cleaner_complete,files_match_report,schema_matches,players_checked,kda_mismatching_players,assumed_death_links,same_side_assists,verified,inspection_error
0,9079737909,True,True,True,10,0,0,0,True,
1,9079737994,True,True,True,10,0,0,0,True,
2,9079738443,True,True,True,10,0,0,0,True,
3,9079738574,True,True,True,10,0,0,1,True,
4,9079740225,True,True,True,10,0,0,0,True,
5,9079740632,True,True,True,10,0,0,0,True,
6,9079741152,True,True,True,10,0,2,0,True,
7,9079743269,True,True,True,10,0,0,0,True,
8,9079745340,True,True,True,10,0,0,0,True,
9,9079746862,True,True,True,10,0,0,0,True,


Matches with verified published credit: 10 / 10
Players checked: 100
Mismatching players: 0 | Not verified: 0


## 8. Final player snapshots and inventory

Each player's last written stats snapshot is shown below. Credit counts in the preceding
sections are independently reconstructed from deaths/assists. Inventory remains a separate
slot-level table, so empty item slots are not mistaken for missing players.


In [22]:
if "players" in tables and "player_stats" in tables:
    stats = tables["player_stats"]
    last_ticks = stats.groupby(["match_id", "player_slot"], as_index=False)["tick"].max()
    final_stats = stats.merge(last_ticks, on=["match_id", "player_slot", "tick"], how="inner")
    roster_view = tables["players"].merge(final_stats, on=["match_id", "player_slot"], how="left", validate="one_to_one")
    display(roster_view.sort_values("player_slot"))
if "final_inventory" in tables:
    display(tables["final_inventory"].sort_values(["player_slot", "inventory_slot"]))

,player_slot,team,team_slot,hero_name,hero_id,match_id,tick,game_time_s,level,kills,deaths,assists,net_worth,unspent_gold,total_earned_gold,total_earned_xp,last_hits,denies,hero_damage,tower_damage,healing
0,0,2,0,npc_dota_hero_juggernaut,8,9079737909,45240,1366,11,1,3,5,7800,235,7504,8166,135,0,1808,0,0
1,1,2,1,npc_dota_hero_legion_commander,104,9079737909,45240,1366,9,3,3,4,6020,205,5887,5534,72,0,5173,0,0
2,2,2,2,npc_dota_hero_tidehunter,29,9079737909,45240,1366,8,4,2,3,5162,347,4847,4350,43,0,4434,0,0
3,3,2,3,npc_dota_hero_axe,2,9079737909,45240,1366,10,2,3,4,5985,945,5661,6289,79,0,2000,0,0
4,4,2,4,npc_dota_hero_sniper,35,9079737909,45240,1366,9,2,8,0,5246,81,5084,5731,49,0,2826,0,0
5,5,3,0,npc_dota_hero_drow_ranger,6,9079737909,45240,1366,8,1,4,5,5905,740,5672,4227,45,0,2736,2879,0
6,6,3,1,npc_dota_hero_sven,18,9079737909,45240,1366,8,2,5,2,5264,449,5089,4315,33,0,5096,2139,0
7,7,3,2,npc_dota_hero_centaur,96,9079737909,45240,1366,9,2,3,2,5980,940,5684,5499,43,0,4638,2915,0
8,8,3,3,npc_dota_hero_huskar,59,9079737909,45240,1366,11,6,2,0,6876,386,6526,7543,39,0,4511,5777,0
9,9,3,4,npc_dota_hero_lina,25,9079737909,45240,1366,8,5,4,1,5458,318,5382,4886,18,0,4822,2318,0


,player_slot,inventory_slot,item_name,match_id
53,0,0,item_bfury,9079737909
0,0,1,item_magic_wand,9079737909
12,0,2,item_foragers_mana,9079737909
1,0,3,item_power_treads,9079737909
34,0,4,item_wraith_band,9079737909
...,...,...,...,...
11,9,1,item_travel_boots,9079737909
73,9,2,item_bottle,9079737909
74,9,3,item_magic_wand,9079737909
32,9,4,item_ogre_axe,9079737909


## 9. Browse a table by player and time

Change these controls and rerun the cell. Player columns vary: `player_slot` in stats/positions/
purchases, `attacker_slot` or `target_slot` in damage, `victim_slot`/`killer_slot`/`credited_killer_slot` in deaths, and
`assist_slot` in assists. Time filters require the corresponding column; assists have raw ticks
but no `game_time_s` column in the warehouse contract.
For scoreboard kill credit, filter deaths by `credited_killer_slot`. `killer_slot` identifies
the direct last hitter. `link_status` and `same_side` remain available in `SHOW_COLUMNS`.


In [23]:
BROWSE_TABLE = "positions"
PLAYER_COLUMN = "player_slot"
PLAYER_SLOT = 4  # None disables player filtering; None PLAYER_COLUMN as well is allowed.
TICK_MIN = None
TICK_MAX = None
GAME_TIME_MIN = None  # Negative values include pregame; 0 starts at the chosen horn snapshot.
GAME_TIME_MAX = None
SHOW_COLUMNS = None  # None for all columns, or a list of names.
LIMIT = 20

if BROWSE_TABLE not in tables:
    print(f"No loaded table: {BROWSE_TABLE}")
else:
    subset = tables[BROWSE_TABLE]
    if PLAYER_SLOT is not None:
        if PLAYER_COLUMN not in subset:
            raise ValueError(f"Choose a valid player column from {list(subset.columns)}")
        subset = subset.loc[subset[PLAYER_COLUMN].eq(PLAYER_SLOT).fillna(False)]
    for field, lower, upper in [("tick", TICK_MIN, TICK_MAX), ("game_time_s", GAME_TIME_MIN, GAME_TIME_MAX)]:
        if lower is not None or upper is not None:
            if field not in subset:
                raise ValueError(f"{BROWSE_TABLE} has no {field} column")
            if lower is not None:
                subset = subset.loc[subset[field].ge(lower).fillna(False)]
            if upper is not None:
                subset = subset.loc[subset[field].le(upper).fillna(False)]
    if "tick" in subset:
        subset = subset.sort_values("tick", kind="stable")
    print(f"{sources[BROWSE_TABLE]} | {len(subset):,} matching rows; showing up to {LIMIT}")
    display(subset[SHOW_COLUMNS].head(LIMIT) if SHOW_COLUMNS is not None else subset.head(LIMIT))

published output | 1,457 matching rows; showing up to 20


,player_slot,tick,game_time_s,x,y,match_id
5828,4,1560,-90,-7100.0,-6150.0,9079737909
5829,4,1590,-89,-7100.0,-6150.0,9079737909
5830,4,1620,-88,-7100.0,-6150.0,9079737909
5831,4,1650,-87,-7100.0,-6150.0,9079737909
5832,4,1680,-86,-7100.0,-6150.0,9079737909
5833,4,1710,-85,-7100.0,-6150.0,9079737909
5834,4,1740,-84,-7100.0,-6150.0,9079737909
5835,4,1770,-83,-7100.0,-6150.0,9079737909
5836,4,1800,-82,-7100.0,-6150.0,9079737909
5837,4,1830,-81,-7100.0,-6150.0,9079737909


## 10. Full saved report

This is the selected match's report, including the credit method, contract version,
window size, findings, errors and completion flags. To change cleaning behavior,
update and rerun the cleaner. This notebook leaves all saved outputs unchanged.


In [24]:
print(json.dumps(report, indent=2) if report is not None else "No report found")

{
  "match_id": 9079737909,
  "table_rows": {
    "matches": 1,
    "players": 10,
    "player_stats": 15080,
    "positions": 14570,
    "damage_events": 8783,
    "purchases": 200,
    "deaths": 37,
    "death_assists": 26,
    "final_inventory": 75
  },
  "findings": {
    "invalid_game_clock": {
      "rows": 0,
      "blocking": true
    },
    "excluded_resource_rows": {
      "rows": 0,
      "blocking": false
    },
    "duplicate_player_snapshots": {
      "rows": 0,
      "blocking": true
    },
    "unmapped_resource_rows": {
      "rows": 0,
      "blocking": true
    },
    "duplicate_team_snapshots": {
      "rows": 0,
      "blocking": true
    },
    "missing_team_snapshots": {
      "rows": 0,
      "blocking": true
    },
    "unknown_replication_marker": {
      "rows": 0,
      "blocking": true
    },
    "invalid_hero_player_ids": {
      "rows": 0,
      "blocking": true
    },
    "unmapped_hero_teams": {
      "rows": 0,
      "blocking": true
    },
    "damage